Load Stage 4 Cleaned Dataset

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

# Upload Stage 4 cleaned dataset
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Stage 4 cleaned dataset loaded successfully!")
print("Dataset Shape:", df.shape)

Saving railway_delay_analyzer_finaldataset to railway_delay_analyzer_finaldataset
Stage 4 cleaned dataset loaded successfully!
Dataset Shape: (50000, 40)


Validate Date Column

In [2]:
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Date validation completed!")

print("\nDate Data Type:")
print(df["date"].dtype)

print("\nMissing Dates:")
print(df["date"].isnull().sum())

print("\nDate Range:")
print("Start:", df["date"].min())
print("End:", df["date"].max())

Date validation completed!

Date Data Type:
datetime64[ns]

Missing Dates:
0

Date Range:
Start: 2025-02-08 00:00:00
End: 2026-02-07 00:00:00


Monthly Delay Analysis using resample()

In [3]:
monthly_delay = (
    df.set_index("date")
      .resample("ME")["delay"]
      .agg(
          average_delay="mean",
          maximum_delay="max",
          minimum_delay="min",
          total_delay="sum",
          delay_records="count"
      )
      .reset_index()
)

print("MONTHLY DELAY ANALYSIS")
print("=" * 70)

display(monthly_delay)

MONTHLY DELAY ANALYSIS


,date,average_delay,maximum_delay,minimum_delay,total_delay,delay_records
0,2025-02-28,99.647562,1930.0,-59.0,198199.0,1989
1,2025-03-31,88.463557,675.0,-76.0,364116.0,4116
2,2025-04-30,82.361934,780.0,-74.0,369558.0,4487
3,2025-05-31,90.245102,722.0,-57.0,442201.0,4900
4,2025-06-30,102.646884,3335.0,-24.0,449696.0,4381
5,2025-07-31,69.511508,660.0,-29.0,295980.0,4258
6,2025-08-31,72.013565,893.0,-36.0,318516.0,4423
7,2025-09-30,66.549793,524.0,-36.0,288693.0,4338
8,2025-10-31,93.987895,664.0,-26.0,341646.0,3635
9,2025-11-30,86.803539,795.0,-33.0,333586.0,3843


Monthly Delayed Records


In [4]:
monthly_delayed = (
    df[df["delay"] > 0]
    .set_index("date")
    .resample("ME")
    .size()
    .reset_index(name="delayed_records")
)

print("MONTHLY DELAYED RECORDS")
print("=" * 60)

display(monthly_delayed)

MONTHLY DELAYED RECORDS


,date,delayed_records
0,2025-02-28,1808
1,2025-03-31,3803
2,2025-04-30,4084
3,2025-05-31,4514
4,2025-06-30,3957
5,2025-07-31,3823
6,2025-08-31,3907
7,2025-09-30,3834
8,2025-10-31,3232
9,2025-11-30,3417


Train-wise Analysis

In [5]:
train_analysis = (
    df.groupby(
        ["train_no", "train_name"]
    )
    .agg(
        total_records=("train_no", "size"),
        average_delay=("delay", "mean"),
        maximum_delay=("delay", "max"),
        total_delay=("delay", "sum"),
        delayed_records=("delay", lambda x: (x > 0).sum())
    )
    .reset_index()
)

# Calculate delayed percentage
train_analysis["delayed_percentage"] = (
    train_analysis["delayed_records"] /
    train_analysis["total_records"] * 100
)

# Sort by average delay
train_analysis = train_analysis.sort_values(
    by="average_delay",
    ascending=False
)

print("TRAIN-WISE DELAY ANALYSIS")
print("=" * 70)

display(train_analysis.head(15))

TRAIN-WISE DELAY ANALYSIS


,train_no,train_name,total_records,average_delay,maximum_delay,total_delay,delayed_records,delayed_percentage
3,1022,Unknown,46,348.826087,561.0,16046.0,46,100.000000
22,1054,Unknown,563,331.028419,1181.0,186369.0,558,99.111901
2,1021,Unknown,46,297.760870,533.0,13697.0,46,100.000000
7,1025,Unknown,861,209.073171,784.0,180012.0,853,99.070848
16,1030,Unknown,127,195.417323,778.0,24818.0,119,93.700787
9,1026,Unknown,2886,191.963964,3335.0,554008.0,2828,97.990298
20,1053,Unknown,862,180.892111,886.0,155929.0,838,97.215777
29,1140,Unknown,118,161.644068,495.0,19074.0,102,86.440678
13,1028,Unknown,4692,158.588662,1930.0,744098.0,4203,89.578005
11,1027,Unknown,1636,156.405257,1855.0,255879.0,1280,78.239609


Station-wise Analysis

In [6]:
station_analysis = (
    df.groupby(
        ["station_no", "station_name"]
    )
    .agg(
        total_records=("station_no", "size"),
        average_delay=("delay", "mean"),
        maximum_delay=("delay", "max"),
        total_delay=("delay", "sum"),
        delayed_records=("delay", lambda x: (x > 0).sum())
    )
    .reset_index()
)

station_analysis["delayed_percentage"] = (
    station_analysis["delayed_records"] /
    station_analysis["total_records"] * 100
)

station_analysis = station_analysis.sort_values(
    by="average_delay",
    ascending=False
)

print("STATION-WISE DELAY ANALYSIS")
print("=" * 70)

display(station_analysis.head(15))

STATION-WISE DELAY ANALYSIS


,station_no,station_name,total_records,average_delay,maximum_delay,total_delay,delayed_records,delayed_percentage
160,10,PCOI,2,648.000000,651.0,1296.0,2,100.0
369,25,LTT,3,589.000000,1130.0,1767.0,3,100.0
364,24,TNA,26,575.923077,1181.0,14974.0,26,100.0
348,23,KYN,26,533.961538,1160.0,13883.0,26,100.0
323,21,NK,26,471.538462,1141.0,12260.0,26,100.0
193,12,PCOI,9,451.888889,1573.0,4067.0,9,100.0
333,22,IGP,26,442.384615,1100.0,11502.0,26,100.0
227,14,PPI,26,418.884615,1050.0,10891.0,26,100.0
55,4,ARMU,2,408.500000,561.0,817.0,2,100.0
207,13,NU,26,408.500000,1034.0,10621.0,26,100.0


Delayed Trains Analysis

In [7]:
delayed_train_analysis = (
    df[df["delay"] > 0]
    .groupby(
        ["train_no", "train_name"]
    )
    .agg(
        delayed_records=("delay", "size"),
        average_delay=("delay", "mean"),
        maximum_delay=("delay", "max"),
        total_delay=("delay", "sum")
    )
    .reset_index()
    .sort_values(
        by="average_delay",
        ascending=False
    )
)

print("DELAYED TRAIN ANALYSIS")
print("=" * 70)

display(delayed_train_analysis.head(15))

DELAYED TRAIN ANALYSIS


,train_no,train_name,delayed_records,average_delay,maximum_delay,total_delay
3,1022,Unknown,46,348.826087,561.0,16046.0
22,1054,Unknown,558,333.994624,1181.0,186369.0
2,1021,Unknown,46,297.760870,533.0,13697.0
7,1025,Unknown,853,211.039859,784.0,180017.0
16,1030,Unknown,119,208.554622,778.0,24818.0
11,1027,Unknown,1280,199.905469,1855.0,255879.0
9,1026,Unknown,2828,195.900990,3335.0,554008.0
29,1140,Unknown,102,187.029412,495.0,19077.0
20,1053,Unknown,838,186.107399,886.0,155958.0
13,1028,Unknown,4203,177.039734,1930.0,744098.0


Special Train Analysis

In [8]:
special_mask = (
    df["type_code"]
    .astype(str)
    .str.contains(
        "SPECIAL",
        case=False,
        na=False
    )
    |
    df["train_name"]
    .astype(str)
    .str.contains(
        r"\bSPL\b|SPECIAL",
        case=False,
        na=False,
        regex=True
    )
)

special_trains = df[special_mask].copy()

print("SPECIAL TRAIN ANALYSIS")
print("=" * 70)

print("Special train records:", len(special_trains))
print(
    "Unique special trains:",
    special_trains["train_no"].nunique())

SPECIAL TRAIN ANALYSIS
Special train records: 34751
Unique special trains: 17


Special Train Delay Analysis

In [9]:
if len(special_trains) > 0:

    special_train_analysis = (
        special_trains
        .groupby(
            ["train_no", "train_name"]
        )
        .agg(
            total_records=("train_no", "size"),
            average_delay=("delay", "mean"),
            maximum_delay=("delay", "max"),
            total_delay=("delay", "sum"),
            delayed_records=(
                "delay",
                lambda x: (x > 0).sum()
            )
        )
        .reset_index()
    )

    special_train_analysis["delayed_percentage"] = (
        special_train_analysis["delayed_records"] /
        special_train_analysis["total_records"] * 100
    )

    special_train_analysis = (
        special_train_analysis
        .sort_values(
            by="average_delay",
            ascending=False
        )
    )

    print("Special train delay analysis:")
    display(special_train_analysis.head(15))

else:

    special_train_analysis = pd.DataFrame()

    print("No special train records found.")

Special train delay analysis:


,train_no,train_name,total_records,average_delay,maximum_delay,total_delay,delayed_records,delayed_percentage
9,1042,SNSI DR SPECIAL,2,146.000000,147.0,292.0,2,100.000000
14,1140,MAO NGP SPECIAL,2600,138.415385,649.0,359880.0,2567,98.730769
6,1027,DR GKP SPECIAL,3375,111.212741,1762.0,375343.0,3235,95.851852
4,1025,DR BALLIA SPL,2797,110.162317,679.0,308124.0,2677,95.709689
11,1054,DNR MMR SPL,48,99.645833,612.0,4783.0,43,89.583333
12,1129,CSMT SWV SPL,15,64.466667,313.0,967.0,13,86.666667
5,1026,BUI DR SPL,751,57.615180,544.0,43269.0,573,76.298269
15,1211,BD NK SPL,5569,41.533669,272.0,231301.0,5089,91.380858
2,1023,PUNE KOP SPECIAL,7964,36.430437,301.0,290132.0,7533,94.588147
7,1028,GKP DR SPL,438,33.522831,1081.0,14683.0,89,20.319635


Weather-wise Delay Analysis

In [10]:
weather_analysis = (
    df.groupby("weather_condition")
      .agg(
          total_records=("delay", "size"),
          average_delay=("delay", "mean"),
          maximum_delay=("delay", "max"),
          total_delay=("delay", "sum")
      )
      .reset_index()
      .sort_values(
          by="average_delay",
          ascending=False
      )
)

print("WEATHER-WISE DELAY ANALYSIS")
print("=" * 70)

display(weather_analysis)

WEATHER-WISE DELAY ANALYSIS


,weather_condition,total_records,average_delay,maximum_delay,total_delay
3,Fog/Mist,1841,103.134709,1903.0,189871.0
2,Cool & Clear,1600,95.869375,1927.0,153391.0
0,Clear,16830,91.014439,3235.0,1531773.0
7,Light Rain,759,90.673254,894.0,68821.0
8,Partly Cloudy,5122,87.584147,757.0,448606.0
5,Hot & Dry,1689,87.522795,718.0,147826.0
1,Cloudy,10822,86.362133,1881.0,934611.0
4,Heavy Rain,2102,80.049477,3335.0,168264.0
9,Rain,6125,79.546286,1141.0,487221.0
6,Humid/Cloudy,3110,77.887781,3141.0,242231.0


Congestion-wise Delay Analysis

In [11]:
congestion_analysis = (
    df.groupby("congestion_level")
      .agg(
          total_records=("delay", "size"),
          average_delay=("delay", "mean"),
          maximum_delay=("delay", "max"),
          total_delay=("delay", "sum")
      )
      .reset_index()
      .sort_values(
          by="average_delay",
          ascending=False
      )
)

print("CONGESTION-WISE DELAY ANALYSIS")
print("=" * 70)

display(congestion_analysis)

CONGESTION-WISE DELAY ANALYSIS


,congestion_level,total_records,average_delay,maximum_delay,total_delay
3,Severe,10035,257.635476,3335.0,2585372.0
0,High,12294,93.966732,1573.0,1155227.0
2,Moderate,26139,23.993726,135.0,627172.0
1,Low,1532,3.161880,38.0,4844.0


Operational Disruption Analysis

In [12]:
disruption_analysis = (
    df.groupby("operational_disruption")
      .agg(
          total_records=("delay", "size"),
          average_delay=("delay", "mean"),
          maximum_delay=("delay", "max"),
          total_delay=("delay", "sum")
      )
      .reset_index()
      .sort_values(
          by="average_delay",
          ascending=False
      )
)

print("OPERATIONAL DISRUPTION ANALYSIS")
print("=" * 70)

display(disruption_analysis)

OPERATIONAL DISRUPTION ANALYSIS


,operational_disruption,total_records,average_delay,maximum_delay,total_delay
1,Major operational disruption,9737,268.670227,3335.0,2616042.0
4,Significant delay / traffic regulation,10707,107.899692,180.0,1155282.0
2,Minor operational delay,15804,34.404961,60.0,543736.0
5,Weather-related operating impact,2303,4.620495,15.0,10641.0
0,Holiday traffic congestion,559,4.409660,15.0,2465.0
3,Normal operations,10890,4.081635,15.0,44449.0


Overall Delay Summary

In [13]:
total_records = len(df)

delayed_records = (
    df["delay"] > 0
).sum()

high_delay_records = (
    df["delay"] > 15
).sum()

average_delay = df["delay"].mean()
maximum_delay = df["delay"].max()

delay_summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Delayed Records",
        "Delay > 15 Minutes",
        "Average Delay",
        "Maximum Delay"
    ],
    "Value": [
        total_records,
        delayed_records,
        high_delay_records,
        average_delay,
        maximum_delay
    ]
})

print("OVERALL DELAY SUMMARY")
print("=" * 60)

display(delay_summary)

OVERALL DELAY SUMMARY


,Metric,Value
0,Total Records,50000.0000
1,Delayed Records,44444.0000
2,Delay > 15 Minutes,35619.0000
3,Average Delay,87.4523
4,Maximum Delay,3335.0000


Save Stage 6 Analysis Results

In [14]:
# Save monthly analysis
monthly_delay.to_csv(
    "stage6_monthly_delay_analysis.csv",
    index=False
)

monthly_delayed.to_csv(
    "stage6_monthly_delayed_records.csv",
    index=False
)

# Save train analysis
train_analysis.to_csv(
    "stage6_train_analysis.csv",
    index=False
)

# Save station analysis
station_analysis.to_csv(
    "stage6_station_analysis.csv",
    index=False
)

# Save delayed train analysis
delayed_train_analysis.to_csv(
    "stage6_delayed_train_analysis.csv",
    index=False
)

# Save special train analysis
if len(special_train_analysis) > 0:
    special_train_analysis.to_csv(
        "stage6_special_train_analysis.csv",
        index=False
    )

# Save weather analysis
weather_analysis.to_csv(
    "stage6_weather_analysis.csv",
    index=False
)

# Save congestion analysis
congestion_analysis.to_csv(
    "stage6_congestion_analysis.csv",
    index=False
)

# Save operational disruption analysis
disruption_analysis.to_csv(
    "stage6_operational_disruption_analysis.csv",
    index=False
)

# Save overall summary
delay_summary.to_csv(
    "stage6_overall_delay_summary.csv",
    index=False
)

print("All Stage 6 analysis files saved successfully!")

All Stage 6 analysis files saved successfully!
